# Week 3: Dual-85 Routed LoRA Experiment

This notebook targets at least 85% on both held-out synthetic facts and untouched general controls.

1. Synthetic questions are answered with the trained LoRA adapter enabled.
2. General questions are answered with the same base model but the adapter disabled.

This task routing prevents synthetic-fact specialization from overwriting ordinary behavior. The notebook also evaluates the adapter with routing disabled as a diagnostic, so the source of capability preservation is explicit.

The 50 final general-control questions are never used for training or epoch selection.


## 0. Runtime Setup

Use a hosted Google Colab GPU runtime. A T4 is sufficient for this notebook.

Run the package cell once in a fresh runtime. If Colab says packages were already imported, restart the runtime before continuing. The Hugging Face token warning is harmless because the Qwen model is public.


In [ ]:
%pip uninstall -q -y bitsandbytes
%pip install -q -U "transformers==4.48.3" "accelerate==1.3.0" "peft==0.14.0" "datasets==3.2.0" sentencepiece "pandas==2.2.3"
%pip install -q --no-deps "bitsandbytes==0.49.2"


## 1. Mount Drive and Set Paths

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
THESIS_DIR = Path('/content/drive/MyDrive/Thesis')

DATA_DIR = THESIS_DIR / 'Week 2' / 'data' / 'synthetic_facts_v1'
CONTROL_DATA_DIR = THESIS_DIR / 'Week 3' / 'data' / 'general_controls_v1'
CONTROL_MANIFEST_PATH = CONTROL_DATA_DIR / 'manifest.json'
GENERAL_CONTROL_PATH = CONTROL_DATA_DIR / 'general_control.jsonl'
GENERAL_REPLAY_PATH = CONTROL_DATA_DIR / 'general_replay.jsonl'
GENERAL_VALIDATION_PATH = CONTROL_DATA_DIR / 'general_validation.jsonl'

CONTROL_DATA_DIR.mkdir(parents=True, exist_ok=True)
required_control_files = [
    CONTROL_MANIFEST_PATH,
    GENERAL_CONTROL_PATH,
    GENERAL_REPLAY_PATH,
    GENERAL_VALIDATION_PATH,
]

missing_control_files = [
    path for path in required_control_files if not path.exists()
]
if missing_control_files:
    from google.colab import files

    print('Upload these fixed files from Week 3/data/general_controls_v1:')
    for path in missing_control_files:
        print('-', path.name)
    uploaded = files.upload()
    for path in missing_control_files:
        assert path.name in uploaded, f'Missing uploaded file: {path.name}'
        path.write_bytes(uploaded[path.name])

for required_file in required_control_files:
    assert required_file.exists(), f'Missing control dataset file: {required_file}'

RUN_DIR = THESIS_DIR / 'Week 3' / 'results' / 'week3_dual85_routed_lora'
ADAPTER_DIR = RUN_DIR / 'adapter'
CHECKPOINT_DIR = RUN_DIR / 'epoch_checkpoints'
OUTPUT_DIR = RUN_DIR / 'results'

TRAIN_PATH = DATA_DIR / 'train_all.jsonl'
EVAL_FORGET_PATH = DATA_DIR / 'eval_forget.jsonl'
EVAL_RETAIN_PATH = DATA_DIR / 'eval_retain.jsonl'

for folder in [RUN_DIR, ADAPTER_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

for required_file in [TRAIN_PATH, EVAL_FORGET_PATH, EVAL_RETAIN_PATH]:
    assert required_file.exists(), (
        f'Missing Week 2 dataset file: {required_file}\n'
        'Run week2_synthetic_facts_dataset.ipynb first.'
    )

print('Thesis folder:', THESIS_DIR)
print('Training data:', TRAIN_PATH)
print('Results folder:', OUTPUT_DIR)


## 2. Imports and Model Configuration

In [ ]:
import gc
import json
import random
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import bitsandbytes as bnb
from bitsandbytes.cextension import lib as bnb_native_lib
from datasets import Dataset
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from peft.tuners.lora import LoraLayer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), (
    'CUDA is unavailable. In Colab select Runtime > Change runtime type > T4 GPU.'
)
assert bnb_native_lib is not None, (
    'bitsandbytes native CUDA library did not load. Restart the Colab runtime, '
    'rerun the package cell, and then continue from the imports cell.'
)

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
SYNTHETIC_SYSTEM_PROMPT = (
    'You answer questions about fictional synthetic people using the provided learned facts.'
)
GENERAL_SYSTEM_PROMPT = (
    'Answer the question concisely. Return only the requested answer without explanation.'
)

# Keep the previously successful learning setup. General capability is
# protected by disabling the adapter for non-synthetic requests.
MAX_LENGTH = 192
MAX_EPOCHS = 20
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
REPLAY_REPETITIONS = 10
SELECTION_SYNTHETIC_PER_SPLIT = 50

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

# Evaluate only mature checkpoints and test weaker adapter strengths too.
SELECTION_EPOCHS = [8, 12, 16, 20]
ADAPTER_STRENGTHS = [0.55, 0.70, 0.85, 1.00]
TARGET_PERCENTAGE = 85.0
MIN_SYNTHETIC_SELECTION_PERCENTAGE = TARGET_PERCENTAGE
# Evaluate all planned checkpoints; do not stop before epoch 20 because the
# previous held-out run needed the later checkpoint to exceed 85% on both splits.
EARLY_STOP_SYNTHETIC_PERCENTAGE = 101.0
EARLY_STOP_GENERAL_PERCENTAGE = TARGET_PERCENTAGE

print('CUDA available:', torch.cuda.is_available())
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('bitsandbytes:', bnb.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Maximum epochs:', MAX_EPOCHS)
print('Learning rate:', LEARNING_RATE)
print('LoRA targets:', LORA_TARGET_MODULES)


## 3. Load Synthetic Evaluation Questions

In [ ]:
def read_jsonl(path):
    records = []
    with Path(path).open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_records = read_jsonl(TRAIN_PATH)
eval_forget_records = read_jsonl(EVAL_FORGET_PATH)
eval_retain_records = read_jsonl(EVAL_RETAIN_PATH)

synthetic_records = []
for split, records in [('forget', eval_forget_records), ('retain', eval_retain_records)]:
    for record in records:
        synthetic_records.append({
            'test_set': 'synthetic',
            'category': record['fact_type'],
            'eval_split': split,
            'example_id': record['example_id'],
            'prompt': record['prompt'],
            'expected_value': str(record['fact_value']),
        })

print('Training examples:', len(train_records))
print('Forget questions:', len(eval_forget_records))
print('Retain questions:', len(eval_retain_records))
print('Total synthetic questions:', len(synthetic_records))


## 4. General Capability Control Questions

These questions are deterministic and avoid current events. Each answer is deliberately short for reliable contains-value scoring.

In [ ]:
import hashlib

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

control_manifest = json.loads(
    CONTROL_MANIFEST_PATH.read_text(encoding='utf-8')
)

for set_name, details in control_manifest['sets'].items():
    dataset_path = CONTROL_DATA_DIR / details['jsonl_file']
    actual_hash = sha256_file(dataset_path)
    assert actual_hash == details['jsonl_sha256'], (
        f'Checksum mismatch for {dataset_path.name}. '
        'The versioned control questions were modified.'
    )

general_records = read_jsonl(GENERAL_CONTROL_PATH)
assert len(general_records) == 50

general_df = pd.DataFrame(general_records)
print('General control questions:', len(general_df))
print('Control dataset SHA-256:', sha256_file(GENERAL_CONTROL_PATH))
display(general_df.groupby('category').size().reset_index(name='num_questions'))


## 5. General Replay and Validation Sets

The replay examples remind the adapter how ordinary questions should be answered. They are different from the 50 final control questions.

The validation questions are also separate from both replay and final controls. They are used only to select the best epoch.


In [ ]:
general_replay_records = read_jsonl(GENERAL_REPLAY_PATH)
general_validation_records = read_jsonl(GENERAL_VALIDATION_PATH)

assert len(general_replay_records) == 30
assert len(general_validation_records) == 20

control_prompts = {record['prompt'].strip().lower() for record in general_records}
replay_prompts = {
    record['prompt'].strip().lower() for record in general_replay_records
}
validation_prompts = {
    record['prompt'].strip().lower() for record in general_validation_records
}

assert control_prompts.isdisjoint(replay_prompts)
assert control_prompts.isdisjoint(validation_prompts)
assert replay_prompts.isdisjoint(validation_prompts)

print('General replay examples:', len(general_replay_records))
print('General validation examples:', len(general_validation_records))
print('Final general controls remain untouched:', len(general_records))


## 6. Evaluation Functions


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def normalize_text(text):
    text = str(text).strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip(' .,!?:;"\'')

def contains_expected_value(generated, expected):
    generated = normalize_text(generated)
    expected = normalize_text(expected)
    if not expected:
        return False
    pattern = rf'(?<!\w){re.escape(expected)}(?!\w)'
    return re.search(pattern, generated) is not None

def load_base_model():
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    loaded_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map='auto',
    )
    loaded_model.eval()
    return loaded_model

def system_prompt_for(test_set):
    return (
        SYNTHETIC_SYSTEM_PROMPT
        if test_set == 'synthetic'
        else GENERAL_SYSTEM_PROMPT
    )

@torch.inference_mode()
def generate_answer(model, prompt, test_set, max_new_tokens=16):
    messages = [
        {'role': 'system', 'content': system_prompt_for(test_set)},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def evaluate(model, records, model_stage, progress_every=100):
    rows = []
    for i, record in enumerate(records, start=1):
        generated = generate_answer(model, record['prompt'], record['test_set'])
        expected = normalize_text(record['expected_value'])
        generated_normalized = normalize_text(generated)
        rows.append({
            **record,
            'model_stage': model_stage,
            'generated_answer': generated,
            'exact_match': generated_normalized == expected,
            'contains_value': contains_expected_value(generated_normalized, expected),
        })
        if progress_every and i % progress_every == 0:
            print(f'{model_stage}: {i}/{len(records)}')
    return pd.DataFrame(rows)


## 7. Test the Untouched Generic Model


In [ ]:
model = load_base_model()

base_synthetic_df = evaluate(model, synthetic_records, 'base_before_training')
base_general_df = evaluate(model, general_records, 'base_before_training')
base_validation_df = evaluate(
    model,
    general_validation_records,
    'base_before_training_validation',
    progress_every=0,
)

base_synthetic_df.to_csv(OUTPUT_DIR / 'base_synthetic_results.csv', index=False)
base_general_df.to_csv(OUTPUT_DIR / 'base_general_control_results.csv', index=False)

print('Base synthetic contains-value:', f"{100 * base_synthetic_df['contains_value'].mean():.2f}%")
print('Base general contains-value:', f"{100 * base_general_df['contains_value'].mean():.2f}%")
print('Base validation contains-value:', f"{100 * base_validation_df['contains_value'].mean():.2f}%")


## 8. Train with Replay and Select Epoch plus Adapter Strength

The previous run underfit because the adapter was too small and too restricted. This run restores the LoRA capacity that successfully learned the synthetic facts, while mixing in 300 replay rows.

Selection checks epochs 8, 12, 16, and 20. At each checkpoint it tests adapter strengths from 0.55 to 1.00. A candidate must reach at least 85% on seen synthetic facts. General validation is measured through the adapter-disabled route. All four checkpoints are evaluated because the previous held-out run needed the later checkpoint to exceed 85% on both synthetic splits.

Selection still uses only:

- seen synthetic training prompts;
- disjoint general validation questions.

The final synthetic paraphrases and 50 general controls remain untouched.


In [ ]:
def make_training_record(record, test_set):
    if test_set == 'synthetic':
        return {
            'test_set': 'synthetic',
            'prompt': record['prompt'],
            'target': str(record['fact_value']),
        }
    return {
        'test_set': 'general_replay',
        'prompt': record['prompt'],
        'target': str(record['expected_value']),
    }

synthetic_training_records = [
    make_training_record(record, 'synthetic')
    for record in train_records
]
replay_training_records = [
    make_training_record(record, 'general_replay')
    for _ in range(REPLAY_REPETITIONS)
    for record in general_replay_records
]
mixed_training_records = synthetic_training_records + replay_training_records
random.Random(SEED).shuffle(mixed_training_records)

def tokenize_train_record(record):
    messages = [
        {'role': 'system', 'content': system_prompt_for(record['test_set'])},
        {'role': 'user', 'content': record['prompt']},
        {'role': 'assistant', 'content': record['target']},
    ]
    prompt_messages = messages[:-1]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    labels = full['input_ids'].copy()
    prompt_len = min(len(prompt['input_ids']), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    if all(label == -100 for label in labels):
        labels[-1] = full['input_ids'][-1]
    full['labels'] = labels
    return full

mixed_train_df = pd.DataFrame(mixed_training_records)
train_dataset = Dataset.from_list(mixed_training_records).map(
    tokenize_train_record,
    remove_columns=list(mixed_train_df.columns),
)

# Use seen training prompts only for epoch selection, never final evaluation prompts.
forget_seen = [
    record for record in train_records if record['split'] == 'forget'
][:SELECTION_SYNTHETIC_PER_SPLIT]
retain_seen = [
    record for record in train_records if record['split'] == 'retain'
][:SELECTION_SYNTHETIC_PER_SPLIT]
synthetic_selection_records = []
for record in forget_seen + retain_seen:
    synthetic_selection_records.append({
        'test_set': 'synthetic',
        'category': record['fact_type'],
        'eval_split': record['split'],
        'example_id': f"selection_{record['example_id']}",
        'prompt': record['prompt'],
        'expected_value': str(record['fact_value']),
    })

model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules=LORA_TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def set_lora_strength(model, strength):
    for module in model.modules():
        if isinstance(module, LoraLayer):
            for adapter_name in list(module.scaling.keys()):
                module.set_scale(adapter_name, strength)

class DualTargetSelectionCallback(TrainerCallback):
    def __init__(self):
        self.rows = []
        self.best_score = -1.0
        self.best_epoch = None
        self.best_strength = None
        self.best_synthetic_percentage = None
        self.best_general_percentage = None

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(round(state.epoch))
        if epoch not in SELECTION_EPOCHS:
            return control

        print(f'\n===== EVALUATING EPOCH {epoch}/{MAX_EPOCHS} =====')
        was_training = model.training
        model.config.use_cache = True
        model.eval()

        epoch_path = CHECKPOINT_DIR / f'epoch_{epoch}'
        set_lora_strength(model, 1.0)
        model.save_pretrained(epoch_path)

        early_stop_candidate_found = False
        for strength in ADAPTER_STRENGTHS:
            set_lora_strength(model, strength)

            synthetic_selection_df = evaluate(
                model,
                synthetic_selection_records,
                f'epoch_{epoch}_strength_{strength}_synthetic',
                progress_every=0,
            )
            # Routing rule: ordinary questions use the untouched base path.
            with model.disable_adapter():
                general_selection_df = evaluate(
                    model,
                    general_validation_records,
                    f'epoch_{epoch}_strength_{strength}_general_routed',
                    progress_every=0,
                )

            synthetic_pct = 100 * synthetic_selection_df['contains_value'].mean()
            general_pct = 100 * general_selection_df['contains_value'].mean()

            if synthetic_pct + general_pct > 0:
                harmonic_score = (
                    2 * synthetic_pct * general_pct
                    / (synthetic_pct + general_pct)
                )
            else:
                harmonic_score = 0.0

            # The adapter must learn the synthetic facts. General validation is
            # measured through the adapter-disabled route and remains disjoint
            # from the final 50-question control set.
            eligible = synthetic_pct >= MIN_SYNTHETIC_SELECTION_PERCENTAGE
            selection_score = harmonic_score if eligible else -1.0
            dual_validation_target_met = (
                synthetic_pct >= TARGET_PERCENTAGE
                and general_pct >= TARGET_PERCENTAGE
            )

            self.rows.append({
                'epoch': epoch,
                'adapter_strength': strength,
                'synthetic_seen_percentage': synthetic_pct,
                'general_validation_percentage': general_pct,
                'harmonic_balance_score': harmonic_score,
                'eligible': eligible,
                'dual_validation_target_met': dual_validation_target_met,
                'selection_score': selection_score,
                'checkpoint_path': str(epoch_path),
            })

            print(
                f'Epoch {epoch}, strength {strength:.2f}: '
                f'synthetic={synthetic_pct:.2f}%, '
                f'general={general_pct:.2f}%, '
                f'balance={harmonic_score:.2f}, eligible={eligible}'
            )

            if selection_score > self.best_score:
                self.best_score = selection_score
                self.best_epoch = epoch
                self.best_strength = strength
                self.best_synthetic_percentage = synthetic_pct
                self.best_general_percentage = general_pct
                set_lora_strength(model, 1.0)
                model.save_pretrained(ADAPTER_DIR)
                tokenizer.save_pretrained(ADAPTER_DIR)
                print('Saved as current best routed adapter.')

            if (
                synthetic_pct >= EARLY_STOP_SYNTHETIC_PERCENTAGE
                and general_pct >= EARLY_STOP_GENERAL_PERCENTAGE
            ):
                early_stop_candidate_found = True

        set_lora_strength(model, 1.0)
        model.config.use_cache = False
        if was_training:
            model.train()
        if early_stop_candidate_found:
            control.should_training_stop = True
            print('Early-stop targets reached; ending training after this epoch.')
        return control

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR / 'trainer_state'),
    seed=SEED,
    num_train_epochs=MAX_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_ratio=0.05,
    weight_decay=0.0,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    logging_steps=10,
    save_strategy='no',
    report_to='none',
    remove_unused_columns=False,
)

selection_callback = DualTargetSelectionCallback()
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
    ),
    callbacks=[selection_callback],
)

train_result = trainer.train()
trainer_metrics = {
    key: float(value)
    for key, value in train_result.metrics.items()
    if isinstance(value, (int, float))
}
train_log_history = trainer.state.log_history

epoch_selection_df = pd.DataFrame(selection_callback.rows)
epoch_selection_df.to_csv(OUTPUT_DIR / 'epoch_selection_metrics.csv', index=False)

best_epoch = selection_callback.best_epoch
best_strength = selection_callback.best_strength
best_score = selection_callback.best_score

if best_epoch is None:
    diagnostic = epoch_selection_df.sort_values(
        ['synthetic_seen_percentage', 'general_validation_percentage'],
        ascending=False,
    ).iloc[0]
    raise RuntimeError(
        'No checkpoint reached the minimum synthetic-learning threshold. '
        f"Best observed synthetic score was {diagnostic['synthetic_seen_percentage']:.2f}% "
        f"at epoch {int(diagnostic['epoch'])}, strength {diagnostic['adapter_strength']:.2f}. "
        'Do not continue to final evaluation with an unlearned adapter.'
    )

print('\nEpoch selection results:')
display(epoch_selection_df)
print('Selected epoch:', best_epoch)
print('Selected adapter strength:', best_strength)
print('Selected synthetic validation:', f'{selection_callback.best_synthetic_percentage:.2f}%')
print('Selected general validation:', f'{selection_callback.best_general_percentage:.2f}%')
print('Selected balance score:', f'{best_score:.2f}')


## 9. Reload the Selected Adapter and Run Final Tests

The selected adapter is reloaded from Drive. Final synthetic and general-control evaluation happens only now.


In [ ]:
del trainer
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = load_base_model()
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
set_lora_strength(model, best_strength)
model.eval()
print('Applied selected adapter strength:', best_strength)

lora_synthetic_df = evaluate(model, synthetic_records, 'routed_lora_after_training')

# Diagnostic only: shows what happens if the specialized adapter is incorrectly
# left enabled for unrelated questions.
lora_general_adapter_on_df = evaluate(
    model,
    general_records,
    'adapter_always_on_diagnostic',
)

# Production/thesis route: disable LoRA for ordinary general questions.
with model.disable_adapter():
    lora_general_df = evaluate(
        model,
        general_records,
        'routed_lora_after_training',
    )

lora_synthetic_df.to_csv(OUTPUT_DIR / 'lora_synthetic_results.csv', index=False)
lora_general_df.to_csv(OUTPUT_DIR / 'lora_general_control_results.csv', index=False)
lora_general_adapter_on_df.to_csv(
    OUTPUT_DIR / 'adapter_always_on_general_diagnostic.csv',
    index=False,
)

print('Routed LoRA synthetic contains-value:', f"{100 * lora_synthetic_df['contains_value'].mean():.2f}%")
print('Routed LoRA general contains-value:', f"{100 * lora_general_df['contains_value'].mean():.2f}%")
print('Always-on adapter general diagnostic:', f"{100 * lora_general_adapter_on_df['contains_value'].mean():.2f}%")


## 10. Percentage Summaries and Before/After CSVs


In [ ]:
all_results_df = pd.concat([
    base_synthetic_df,
    base_general_df,
    lora_synthetic_df,
    lora_general_df,
    lora_general_adapter_on_df,
], ignore_index=True)

summary_df = (
    all_results_df
    .groupby(['model_stage', 'test_set', 'eval_split'])
    .agg(
        num_questions=('contains_value', 'size'),
        num_contains_value_correct=('contains_value', 'sum'),
        contains_value_percentage=('contains_value', lambda x: 100 * x.mean()),
        exact_match_percentage=('exact_match', lambda x: 100 * x.mean()),
    )
    .reset_index()
)

category_summary_df = (
    all_results_df
    .groupby(['model_stage', 'test_set', 'eval_split', 'category'])
    .agg(
        num_questions=('contains_value', 'size'),
        num_contains_value_correct=('contains_value', 'sum'),
        contains_value_percentage=('contains_value', lambda x: 100 * x.mean()),
        exact_match_percentage=('exact_match', lambda x: 100 * x.mean()),
    )
    .reset_index()
)

comparison_df = all_results_df.pivot_table(
    index=[
        'test_set',
        'eval_split',
        'example_id',
        'category',
        'prompt',
        'expected_value',
    ],
    columns='model_stage',
    values=['generated_answer', 'exact_match', 'contains_value'],
    aggfunc='first',
).reset_index()
comparison_df.columns = [
    '_'.join(str(part) for part in column if part).strip('_')
    if isinstance(column, tuple) else column
    for column in comparison_df.columns
]

all_results_df.to_csv(OUTPUT_DIR / 'all_test_results.csv', index=False)
summary_df.to_csv(OUTPUT_DIR / 'percentage_summary.csv', index=False)
category_summary_df.to_csv(OUTPUT_DIR / 'percentage_by_category.csv', index=False)
comparison_df.to_csv(OUTPUT_DIR / 'before_after_comparison.csv', index=False)

print('Overall percentage summary:')
display(summary_df)
print('Percentage by category:')
display(category_summary_df)


## 11. Save Metrics JSON and Print Final Percentages


In [ ]:
def percentage(df, split=None):
    subset = df if split is None else df[df['eval_split'] == split]
    return float(100 * subset['contains_value'].mean())

metrics = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_name': 'week3_dual85_routed_lora',
    'base_model_id': MODEL_ID,
    'adapter_dir': str(ADAPTER_DIR),
    'selection': {
        'selected_epoch': int(best_epoch),
        'selected_adapter_strength': float(best_strength),
        'selected_synthetic_validation_percentage': float(selection_callback.best_synthetic_percentage),
        'selected_general_validation_percentage': float(selection_callback.best_general_percentage),
        'selected_balance_score': float(best_score),
        'minimum_synthetic_selection_percentage': MIN_SYNTHETIC_SELECTION_PERCENTAGE,
        'selection_method': 'synthetic threshold plus harmonic balance; general validation evaluated with adapter disabled',
        'final_control_used_for_selection': False,
        'synthetic_eval_used_for_selection': False,
    },
    'training': {
        'num_synthetic_train_examples': len(synthetic_training_records),
        'num_general_replay_examples_before_repetition': len(general_replay_records),
        'replay_repetitions': REPLAY_REPETITIONS,
        'num_mixed_train_rows': len(mixed_training_records),
        'max_epochs': MAX_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,
        'lora_target_modules': LORA_TARGET_MODULES,
        'selection_epochs': SELECTION_EPOCHS,
        'adapter_strengths_tested': ADAPTER_STRENGTHS,
        'trainer_metrics': trainer_metrics,
        'trainer_log_history': train_log_history,
    },
    'num_general_control_questions': len(general_records),
    'base_before_training': {
        'synthetic_forget_contains_value_percentage': percentage(base_synthetic_df, 'forget'),
        'synthetic_retain_contains_value_percentage': percentage(base_synthetic_df, 'retain'),
        'general_control_contains_value_percentage': percentage(base_general_df),
    },
    'routed_lora_after_training': {
        'synthetic_forget_contains_value_percentage': percentage(lora_synthetic_df, 'forget'),
        'synthetic_retain_contains_value_percentage': percentage(lora_synthetic_df, 'retain'),
        'general_control_contains_value_percentage': percentage(lora_general_df),
    },
    'adapter_always_on_diagnostic': {
        'general_control_contains_value_percentage': percentage(lora_general_adapter_on_df),
    },
    'routing_policy': {
        'synthetic_questions': 'LoRA adapter enabled',
        'general_questions': 'LoRA adapter disabled; untouched base-model path',
        'merged_model_exported': False,
    },
}

target_rows = [
    {
        'metric': 'synthetic_forget_contains_value_percentage',
        'percentage': metrics['routed_lora_after_training']['synthetic_forget_contains_value_percentage'],
    },
    {
        'metric': 'synthetic_retain_contains_value_percentage',
        'percentage': metrics['routed_lora_after_training']['synthetic_retain_contains_value_percentage'],
    },
    {
        'metric': 'general_control_contains_value_percentage',
        'percentage': metrics['routed_lora_after_training']['general_control_contains_value_percentage'],
    },
]
target_achievement_df = pd.DataFrame(target_rows)
target_achievement_df['target_percentage'] = TARGET_PERCENTAGE
target_achievement_df['target_met'] = (
    target_achievement_df['percentage'] >= TARGET_PERCENTAGE
)
all_targets_met = bool(target_achievement_df['target_met'].all())
metrics['target_assessment'] = {
    'target_percentage': TARGET_PERCENTAGE,
    'all_targets_met': all_targets_met,
    'requires_forget_retain_and_general_each_to_pass': True,
}

(OUTPUT_DIR / 'metrics.json').write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
(ADAPTER_DIR / 'routing_config.json').write_text(
    json.dumps({
        'base_model_id': MODEL_ID,
        'selected_adapter_strength': float(best_strength),
        'synthetic_route': 'adapter_enabled',
        'general_route': 'adapter_disabled',
        'do_not_merge_adapter': True,
    }, indent=2),
    encoding='utf-8',
)
target_achievement_df.to_csv(
    OUTPUT_DIR / 'target_achievement.csv',
    index=False,
)

print('FINAL CONTAINS-VALUE PERCENTAGES')
print('-' * 50)
print(f"Base synthetic forget: {metrics['base_before_training']['synthetic_forget_contains_value_percentage']:.2f}%")
print(f"Base synthetic retain: {metrics['base_before_training']['synthetic_retain_contains_value_percentage']:.2f}%")
print(f"Base general controls: {metrics['base_before_training']['general_control_contains_value_percentage']:.2f}%")
print()
print(f"Routed LoRA synthetic forget: {metrics['routed_lora_after_training']['synthetic_forget_contains_value_percentage']:.2f}%")
print(f"Routed LoRA synthetic retain: {metrics['routed_lora_after_training']['synthetic_retain_contains_value_percentage']:.2f}%")
print(f"Routed LoRA general controls: {metrics['routed_lora_after_training']['general_control_contains_value_percentage']:.2f}%")
print(f"Adapter always-on general diagnostic: {metrics['adapter_always_on_diagnostic']['general_control_contains_value_percentage']:.2f}%")
print()
display(target_achievement_df)
print('ALL THREE 85% TARGETS MET:', all_targets_met)
print()
print('Selected epoch:', best_epoch)
print('Selected adapter strength:', best_strength)
print('Saved all outputs to:', OUTPUT_DIR)


## Output Files

The notebook saves the routed dual-target experiment under:

`/content/drive/MyDrive/Thesis/Week 3/results/week3_dual85_routed_lora`

- `adapter/`: selected full-strength adapter weights
- `adapter/routing_config.json`: required enable/disable policy and selected strength
- `epoch_checkpoints/`: evaluated checkpoints from epochs 8, 12, 16, and 20
- `results/epoch_selection_metrics.csv`: every epoch/strength trade-off
- `results/base_synthetic_results.csv`
- `results/base_general_control_results.csv`
- `results/lora_synthetic_results.csv`
- `results/lora_general_control_results.csv`
- `results/adapter_always_on_general_diagnostic.csv`
- `results/all_test_results.csv`
- `results/before_after_comparison.csv`
- `results/percentage_summary.csv`
- `results/percentage_by_category.csv`
- `results/target_achievement.csv`
- `results/metrics.json`

The selected adapter strength and routing policy are stored in `metrics.json`. Later notebooks must enable the adapter for synthetic-fact requests and disable it for ordinary general requests.

## Versioned Control Data

The notebook reads fixed files from:

`/content/drive/MyDrive/Thesis/Week 3/data/general_controls_v1`

It verifies their SHA-256 hashes from `manifest.json`. The final 50-question control set is never included in training or checkpoint selection.
